In [37]:
import requests
import os
import urllib
import pandas as pd
import numpy as np

import json
from tqdm.autonotebook import tqdm

import dask.dataframe as dd
from dask.multiprocessing import get
from dask.diagnostics import ProgressBar

from datetime import datetime

from IPython.display import display

tqdm.pandas()

In [38]:
# Potential improvement: strip + upper to reduce the number of unique addresses
# For MW: ~3% less addresses, but cannot be done inplace, otherwise reconciliation wont work.

# Parameters

In [39]:
# Field names required by bePelias. Do not change!
street_field  = "streetName"
housenbr_field = "houseNumber"
postcode_field = "postCode"
city_field  =    "postName"
country_field =  "countryName"


In [40]:
# IP:port of bePelias instance
ws_hostname = "172.27.0.64:4001"

# Parameter to update to your dataset --> replace keys by your input file column names

field_mapping = {
    "streetName": street_field,
    "houseNumber": housenbr_field,
    "postCode": postcode_field,
    "municipalityName": city_field}

data_dir="data/geocoding/mw2"
# data_dir="data/geocoding/bepelias_batch/"
# data_dir="data/geocoding/mw2_20240913/"
data_dir="data/geocoding/mw2_20240913_bis/"  # split
data_dir="data/geocoding/mw2_20240913_ter/"  # no split
data_dir="data/geocoding/mw2_20240913_quatro/"  # split (2d)

input_filename = "addresses_mw2.csv.gz" # Has to be a csv or csv.gz file
input_filename = "addresses.csv" # Has to be a csv or csv.gz file
# input_filename = "sample.csv.gz"

# If you only want to geocode a part of the file, add a query filter (sent to "query()" pandas function)
# filter_query = "countryNisCode.isnull() | (countryNisCode==150)"
filter_query = "countryNisCode.isnull() | (countryNisCode=='150.0')"


# Applyed just before data is sent to geocoded, on the whole dataframe
replace_dict = {"unknown": ""}

In [41]:
# addresses.replace(replace)

In [42]:
cache_filename = "cache.pkl.gz" # Will write bePelias result every "cache_chunk_size" record. Allow to not restart from scratch in case of crash
cache_chunk_size = 10000

In [43]:
output_filename = input_filename.split(".", maxsplit=1)
output_filename = f'{output_filename[0]}_output.{output_filename[1]}'
print(f"Output filename: {output_filename}")

Output filename: addresses_output.csv


# Functions

In [55]:
def call_ws(addr_data, mode="advanced"): #lg = "en,fr,nl"
    t = datetime.now()
    
    fields = { 
            "mode": mode
        }

    if isinstance(addr_data, pd.Series):
        addr_data = addr_data.to_dict()
        
    try: 
        r = requests.get(
        f'http://{ws_hostname}/REST/bepelias/v1/geocode',
            params=addr_data)
        

    except Exception as e:
        print("Exception !")
        print(addr_data)
        print(e)
        raise e
        
    if r.status_code == 204:
        # print("No result!")
        # print(addr_data)
        # print(r.text)
        return
    elif r.status_code == 400:
        print("Argument error")
        print(r.text)
    elif r.status_code == 200:
        try:
            res = json.loads(r.text)
            res["time"] = (datetime.now() - t).total_seconds()
        except ValueError as ve:

            print("Cannot decode result:")
            print(ve)
            print(r.text)
            return r.text
        except AttributeError as ae:
            print(ae)
            print(type(r.text))
            print(r.text)
        return res
    else: 
        print(f"Unknown return code: {r.status_code} ")
        print(r.text)
        print(addr_data)



In [45]:
def call_ws_by_id(best_id): #lg = "en,fr,nl"
    t = datetime.now()
    

       
    try: 
        url = f'http://{ws_hostname}/REST/bepelias/v1/id/{urllib.parse.quote_plus(urllib.parse.quote_plus(best_id))}'
        print(url)
        r = requests.get(url)
        

    except Exception as e:
        print("Exception !")
        print(best_id)
        print(e)
        raise e
        
    if r.status_code == 204:
        # print("No result!")
        # print(addr_data)
        # print(r.text)
        return
    elif r.status_code == 400:
        print("Argument error")
        print(r.text)
    elif r.status_code == 200:
        try:
            res = json.loads(r.text)
            # res["time"] = (datetime.now() - t).total_seconds()
        except ValueError as ve:

            print("Cannot decode result:")
            print(ve)
            print(r.text)
            return r.text
        except AttributeError as ae:
            print(ae)
            print(type(r.text))
            print(r.text)
        return res
    else: 
        print(f"Unknown return code: {r.status_code} ")
        print(r.text)
        



# Calls

## Read data

In [46]:
addresses_filename = f"{data_dir}/{input_filename}"
addresses = pd.read_csv(addresses_filename, dtype=str)
addresses

,streetName,juridicalDistrict,addressType,houseNumber,postCode,id,municipalityName,countryNisCode,targetId,boxNumber
0,Rue Saint-Hubert(DV),910,legal,84,5100,416afc98-9bc4-46a7-b76d-a24295a695d2,Namur,150.0,V0It0ooBL0HW2X5F5T_J,NaN
1,Rue Saint-Hubert(DV),NaN,establishmentUnit,84,5100,2a0099c0-ad23-4afc-acca-e12f9d02027f,Namur,NaN,V0It0ooBL0HW2X5F5T_J,NaN
2,Pladijsstraat,520,legal,2,8540,7195703d-7c33-4c91-abd1-30c8e15e49d6,Deerlijk,150.0,muT30YoBc6EOG49bA0Mq,NaN
3,Vichtestraat(O),NaN,establishmentUnit,31A,8553,2e0ca445-a01e-4386-96e1-17a451644394,Zwevegem,NaN,muT30YoBc6EOG49bA0Mq,NaN
4,Pladijsstraat,NaN,establishmentUnit,2,8540,b11e9038-3a48-4aae-9b11-7ecfc7d982cd,Deerlijk,NaN,muT30YoBc6EOG49bA0Mq,NaN
...,...,...,...,...,...,...,...,...,...,...
1398632,Place Jean Absil(B.S.),NaN,establishmentUnit,10,7603,4ffec516-c350-447c-80c6-b65b83fbb6a5,Péruwelz,NaN,wZhd0ooBUOVbPdvwYZPD,NaN
1398633,Meensesteenweg,520,legal,714,8800,e942e7d9-57a6-4f0c-afae-4f15c0345a1a,Roeselare,150.0,5uRZ1IoBc6EOG49bItVN,NaN
1398634,Meensesteenweg,NaN,establishmentUnit,389,8800,d90f2cf9-8028-4ee6-89f1-ec56d097cccc,Roeselare,NaN,5uRZ1IoBc6EOG49bItVN,NaN
1398635,Avenue Sergent Vrithoff,910,legal,129,5000,e717f0cb-be91-4da0-91de-a4fa82b08265,Namur,150.0,x-WO1YoBc6EOG49byh6j,4


In [47]:
print(f"Input dataset: {addresses.shape[0]} rows")

Input dataset: 1398637 rows


In [48]:
if filter_query:
    addresses = addresses.query(filter_query)
    print(f"After filtering: {addresses.shape[0]} rows")
addresses    

After filtering: 1391302 rows


,streetName,juridicalDistrict,addressType,houseNumber,postCode,id,municipalityName,countryNisCode,targetId,boxNumber
0,Rue Saint-Hubert(DV),910,legal,84,5100,416afc98-9bc4-46a7-b76d-a24295a695d2,Namur,150.0,V0It0ooBL0HW2X5F5T_J,NaN
1,Rue Saint-Hubert(DV),NaN,establishmentUnit,84,5100,2a0099c0-ad23-4afc-acca-e12f9d02027f,Namur,NaN,V0It0ooBL0HW2X5F5T_J,NaN
2,Pladijsstraat,520,legal,2,8540,7195703d-7c33-4c91-abd1-30c8e15e49d6,Deerlijk,150.0,muT30YoBc6EOG49bA0Mq,NaN
3,Vichtestraat(O),NaN,establishmentUnit,31A,8553,2e0ca445-a01e-4386-96e1-17a451644394,Zwevegem,NaN,muT30YoBc6EOG49bA0Mq,NaN
4,Pladijsstraat,NaN,establishmentUnit,2,8540,b11e9038-3a48-4aae-9b11-7ecfc7d982cd,Deerlijk,NaN,muT30YoBc6EOG49bA0Mq,NaN
...,...,...,...,...,...,...,...,...,...,...
1398632,Place Jean Absil(B.S.),NaN,establishmentUnit,10,7603,4ffec516-c350-447c-80c6-b65b83fbb6a5,Péruwelz,NaN,wZhd0ooBUOVbPdvwYZPD,NaN
1398633,Meensesteenweg,520,legal,714,8800,e942e7d9-57a6-4f0c-afae-4f15c0345a1a,Roeselare,150.0,5uRZ1IoBc6EOG49bItVN,NaN
1398634,Meensesteenweg,NaN,establishmentUnit,389,8800,d90f2cf9-8028-4ee6-89f1-ec56d097cccc,Roeselare,NaN,5uRZ1IoBc6EOG49bItVN,NaN
1398635,Avenue Sergent Vrithoff,910,legal,129,5000,e717f0cb-be91-4da0-91de-a4fa82b08265,Namur,150.0,x-WO1YoBc6EOG49byh6j,4


In [49]:
# addresses = addresses.sample(100000, random_state=1)

In [50]:
addresses_unique = addresses[field_mapping.keys()].drop_duplicates()
print(f"Unique addresses : {addresses_unique.shape[0]} rows")
addresses_unique

Unique addresses : 452129 rows


,streetName,houseNumber,postCode,municipalityName
0,Rue Saint-Hubert(DV),84,5100,Namur
2,Pladijsstraat,2,8540,Deerlijk
3,Vichtestraat(O),31A,8553,Zwevegem
5,Heidebaan,90,9100,Sint-Niklaas
7,Jules Maloulaan,26,1040,Etterbeek
...,...,...,...,...
1398630,Hoekstraat,34,9260,Wichelen
1398631,Place Jean Absil(B.S.),10,7603,Péruwelz
1398633,Meensesteenweg,714,8800,Roeselare
1398634,Meensesteenweg,389,8800,Roeselare


In [51]:
try: 
    addresses_geocoded = pd.read_pickle(f"{data_dir}/{cache_filename}")
except FileNotFoundError:
    addresses_geocoded = pd.DataFrame(columns =addresses_unique.columns )
    addresses_geocoded["json"]=pd.NA
addresses_geocoded   

,streetName,houseNumber,postCode,municipalityName,json
0,Rue Royale,100,1000,Bruxelles,"{'geocoding': {'version': '0.2', 'attribution'..."
1,Emmanuellaan,15,1830,Machelen (Brab.),"{'geocoding': {'version': '0.2', 'attribution'..."
2,"Lindenlaan 12, 8660, De Panne",12,8660,De Panne,"{'geocoding': {'version': '0.2', 'attribution'..."
3,Groenenborgerlaan,149,2020,Antwerpen,"{'geocoding': {'version': '0.2', 'attribution'..."
4,Rue du Ruisseau,92,4000,Liège,"{'geocoding': {'version': '0.2', 'attribution'..."
...,...,...,...,...,...
71099,Rue de Courtrai(TOU),39,7500,Tournai,"{'geocoding': {'version': '0.2', 'attribution'..."
71100,Rue des Prés,79,4802,Verviers,"{'geocoding': {'version': '0.2', 'attribution'..."
71101,Hulsterweg (C),111,3980,Tessenderlo,"{'geocoding': {'version': '0.2', 'attribution'..."
71102,Steenbeekpad,1,8880,Sint-Eloois-Winkel,"{'geocoding': {'version': '0.2', 'attribution'..."


In [52]:
addresses_to_geocode = addresses_unique.merge(addresses_geocoded, indicator=True, how="left")
print(f"Found {addresses_to_geocode[addresses_to_geocode._merge == 'both'].shape[0]} addresses in cache")
addresses_to_geocode = addresses_to_geocode[addresses_to_geocode._merge =='left_only']
addresses_to_geocode = addresses_to_geocode.drop(columns=["json", "_merge"])
addresses_to_geocode

Found 71104 addresses in cache


,streetName,houseNumber,postCode,municipalityName
0,Rue Saint-Hubert(DV),84,5100,Namur
1,Pladijsstraat,2,8540,Deerlijk
2,Vichtestraat(O),31A,8553,Zwevegem
3,Heidebaan,90,9100,Sint-Niklaas
4,Jules Maloulaan,26,1040,Etterbeek
...,...,...,...,...
452123,Goudbergstraat,9,8560,Wevelgem
452124,Hoekstraat,34,9260,Wichelen
452126,Meensesteenweg,714,8800,Roeselare
452127,Meensesteenweg,389,8800,Roeselare


In [53]:
chunks = [addresses_to_geocode.iloc[i:i+cache_chunk_size] for i in range(0,len(addresses_to_geocode),cache_chunk_size)]
print(f"{len(chunks)} chunks of size {cache_chunk_size}")


39 chunks of size 10000


# Geocode

In [57]:
initial_start = datetime.now()
for chunk in tqdm(chunks):
    chunk_start = datetime.now()
    dd_addresses = dd.from_pandas(chunk.replace(replace_dict).rename(columns=field_mapping).fillna(""), 
                                  npartitions=min(64, chunk.shape[0]))

    dask_task = dd_addresses[[street_field, housenbr_field, postcode_field, city_field]].apply(call_ws, meta=('x', 'str'), axis=1)

    with ProgressBar(): 
        chunk["json"] = dask_task.compute()


    chunk_time = (datetime.now() - chunk_start).total_seconds()

    ips=chunk.shape[0]/chunk_time

    print(f"{chunk_time:.2f} seconds, {ips:.2f} it/s, {ips*3600:.0f} it/h")

    addresses_geocoded = pd.concat([addresses_geocoded, chunk]).drop_duplicates(subset=addresses_geocoded.drop("json", axis=1).columns)
    
    # To reduce the risk of loosing cache of previous blocs if process crashes during write, 
    # we write a temp file (_cache.pkl.gz) and move it to the final file (cache.pkl.gz) afterward
    addresses_geocoded.to_pickle(f"{data_dir}/_{cache_filename}")
    os.rename(f"{data_dir}/_{cache_filename}", f"{data_dir}/{cache_filename}")
    
    
    

total_time = (datetime.now() - initial_start).total_seconds()

ips=addresses_to_geocode.shape[0]/total_time

print(f"Global : {total_time:.2f} seconds, {ips:.2f} it/s, {ips*3600:.0f} it/h")


  0%|          | 0/39 [00:00<?, ?it/s]

[########################################] | 100% Completed | 177.81 s
178.69 seconds, 55.96 it/s, 201462 it/h


  3%|▎         | 1/39 [03:18<2:05:57, 198.87s/it]

[########################################] | 100% Completed | 205.53 s
205.66 seconds, 48.62 it/s, 175043 it/h


  5%|▌         | 2/39 [07:05<2:12:51, 215.45s/it]

...

[########################################] | 100% Completed | 196.11 s
196.23 seconds, 50.96 it/s, 183454 it/h


 28%|██▊       | 11/39 [40:18<1:45:27, 225.99s/it]

[########################################] | 100% Completed | 205.29 s
205.44 seconds, 48.68 it/s, 175237 it/h


 28%|██▊       | 11/39 [43:45<1:51:22, 238.68s/it]


KeyboardInterrupt: 

In [104]:
# os.rename(f"{data_dir}/_{cache_filename}", f"{data_dir}/{cache_filename}")

In [58]:
addresses_geocoded#.drop_duplicates(subset=addresses_geocoded.drop(["json", "geom"], axis=1).columns)

,streetName,houseNumber,postCode,municipalityName,json
0,Rue Royale,100,1000,Bruxelles,"{'geocoding': {'version': '0.2', 'attribution'..."
1,Emmanuellaan,15,1830,Machelen (Brab.),"{'geocoding': {'version': '0.2', 'attribution'..."
2,"Lindenlaan 12, 8660, De Panne",12,8660,De Panne,"{'geocoding': {'version': '0.2', 'attribution'..."
3,Groenenborgerlaan,149,2020,Antwerpen,"{'geocoding': {'version': '0.2', 'attribution'..."
4,Rue du Ruisseau,92,4000,Liège,"{'geocoding': {'version': '0.2', 'attribution'..."
...,...,...,...,...,...
156605,Gareelmakersstraat,23,2200,Herentals,{'items': [{'bestId': 'https://data.vlaanderen...
156606,Avenue de l'Exposition,384,1090,Jette,{'items': [{'bestId': 'https://databrussels.be...
156607,M. Scheperslaan,174,3550,Heusden-Zolder,{'items': [{'streetname': {'nl': 'M. Schepersl...
156608,"Route du Croisé,Noirefontaine",16,6831,Bouillon,{'items': [{'bestId': 'geodata.wallonie.be/id/...


In [60]:
addresses_geocoded.loc[156609].json

{'items': [{'bestId': 'geodata.wallonie.be/id/Address/189194/1',
   'streetname': {'fr': 'Rue Saint-Pierre'},
   'municipalityName': {'fr': 'Erezée'},
   'partOfMunicipalityName': {'fr': 'Biron'},
   'NIS': '83013',
   'municipalityId': 'geodata.wallonie.be/id/Municipality/83013/7',
   'partOfMunicipalityId': 'geodata.wallonie.be/id/PartOfMunicipality/1786/1',
   'streetId': 'geodata.wallonie.be/id/Streetname/7743602/1',
   'housenumber': '5',
   'status': 'current',
   'precision': 'address',
   'coordinates': [5.48333, 50.30792]},
  {'bestId': 'geodata.wallonie.be/id/Address/189195/1',
   'streetname': {'fr': 'Rue Saint-Pierre'},
   'municipalityName': {'fr': 'Erezée'},
   'partOfMunicipalityName': {'fr': 'Biron'},
   'NIS': '83013',
   'municipalityId': 'geodata.wallonie.be/id/Municipality/83013/7',
   'partOfMunicipalityId': 'geodata.wallonie.be/id/PartOfMunicipality/1786/1',
   'streetId': 'geodata.wallonie.be/id/Streetname/7743602/1',
   'housenumber': '5A',
   'status': 'current

# Reconciliate

In [22]:
def get(dct, keys):
    """
    Get an item of "dct" (dict), by going through "keys", or None 
    Example : 
    keys = ["a", "b", "c"] ==> will return dct["a"]["b"]["c"], or None in any key of the sequence does not exists

    Parameters
    ----------
    dct : dict
        Dictionnary
    keys : list

    Returns
    -------
    An element of dct or None.
    """
    
    for k in keys:
        try: 
            if  dct is None:
                return None
            dct = dct[k]
        except KeyError:
            return None
    return dct

In [23]:
output = addresses_geocoded.copy()
output["best_id"]   = output.json.apply(lambda r: get(r, ["features", 0, "properties", 'addendum', 'best', 'best_id']) or \
                                        get(r, ["features", 0, "properties", 'addendum', 'best', 'street_id']) or \
                                        get(r, ["features", 0, "properties", 'addendum', 'best', 'municipality_id']))
output["geom"]      = output.json.apply(lambda r: get(r, ["features", 0, "geometry","coordinates"] ))
output["precision"] = output.json.apply(lambda r: get(r, ["bepelias", "precision"]) or  "[none]")

for fld in ["streetname_fr", "streetname_nl", "streetname_de", "municipality_name_fr", "municipality_name_nl", "municipality_name_de"]:
    output[fld] = output.json.apply(lambda r: get(r, ["features", 0, "properties", 'addendum', 'best', fld]))
    
for fld in ['housenumber', "postalcode"]:
    output[fld] = output.json.apply(lambda r: get(r, ["features", 0, "properties", fld]))

output = addresses.merge(output.drop(columns="json"))

output.to_csv(f"{data_dir}/{output_filename}", index=False)
output

,streetName,juridicalDistrict,addressType,houseNumber,postCode,id,municipalityName,countryNisCode,targetId,boxNumber,...,geom,precision,streetname_fr,streetname_nl,streetname_de,municipality_name_fr,municipality_name_nl,municipality_name_de,housenumber,postalcode
0,Rue Royale,NaN,worksite,100,1000,6f0c6b2d-1ead-44e2-9135-6e8bbcbf8fc8,Bruxelles,NaN,UW5H1IoBmx0tQqS2amR0,NaN,...,"[4.36283, 50.84808]",address,Rue Royale,Koningsstraat,None,Bruxelles,Brussel,None,100,1000
1,Emmanuellaan,NaN,establishmentUnit,15,1830,36c6962a-b0dc-4889-9967-6a3c5ab84889,Machelen (Brab.),NaN,cm6S04oBmx0tQqS2Mzlj,NaN,...,"[4.41776, 50.90755]",address,None,Emmanuellaan,None,None,Machelen,None,15,1830
2,"Lindenlaan 12, 8660, De Panne",NaN,worksite,12,8660,fb7f2225-8025-4b69-8bc7-85fc23555e6f,De Panne,NaN,nOPe0YoBc6EOG49b0JKG,NaN,...,"[2.58648, 51.09587]",address,None,Lindenlaan,None,La Panne,De Panne,De Panne,12,8660
3,Groenenborgerlaan,NaN,worksite,149,2020,35acce68-73dc-4eb6-b603-c0a663d36153,Antwerpen,NaN,TUIP1YoBL0HW2X5FO9zA,NaN,...,"[4.41331, 51.177]",address,None,Groenenborgerlaan,None,Anvers,Antwerpen,Antwerpen,149,2020
4,Rue du Ruisseau,720,legal,92,4000,d1f9c8b3-80c3-416b-9959-2fa673f9b283,Liège,150.0,hULw0YoBL0HW2X5FOzEy,21,...,"[5.5944, 50.65364]",address,Rue du Ruisseau,None,None,Liège,None,None,92,4000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,Hulsterweg (C),NaN,establishmentUnit,111,3980,87ce627f-035d-458d-8751-f86999e72d24,Tessenderlo,NaN,_EJC0ooBL0HW2X5F1UQz,NaN,...,"[5.12306, 51.06944]",address,None,Hulsterweg,None,None,Tessenderlo,None,111,3980
99996,Rue de la Spinette,NaN,worksite,44,5140,37e7f4eb-b2e8-494c-90b6-87b54c1d7e2a,Sombreffe,NaN,k0KZ1IoBL0HW2X5FLcQJ,NaN,...,"[4.59483, 50.48738]",address,Rue de la Spinette,None,None,Sombreffe,None,None,44,5140
99997,Steenbeekpad,NaN,worksite,1,8880,bb0e2dc8-11d4-4740-bd86-3e30af7e9e14,Sint-Eloois-Winkel,NaN,fW0U0ooBmx0tQqS2vd4C,NaN,...,"[3.19423, 50.86358]",address,None,Steenbeekpad,None,None,Ledegem,None,1,8880
99998,de Bavaylei,NaN,worksite,116,1800,dd293002-c471-4c46-a0e8-67247afa2842,Vilvoorde,NaN,nkKY0ooBL0HW2X5FglbB,NaN,...,"[4.44053, 50.9351]",address,None,de Bavaylei,None,Vilvorde,Vilvoorde,Vilvoorde,116,1800


In [27]:
# output = addresses_geocoded.copy()
# output.json.iloc[1]
# output_filename
output[output.precision=="address_interpol"]

,streetName,juridicalDistrict,addressType,houseNumber,postCode,id,municipalityName,countryNisCode,targetId,boxNumber,...,geom,precision,streetname_fr,streetname_nl,streetname_de,municipality_name_fr,municipality_name_nl,municipality_name_de,housenumber,postalcode
78,"Rue de Saint-Hubert,Redu",820,legal,17,6890,1a26b120-d03c-4acc-b221-25ceaf9aecd1,Libin,150.0,Am6s1IoBmx0tQqS26Hxn,NaN,...,"[5.1604826, 50.0070869]",address_interpol,Rue de Saint-Hubert,None,None,Libin,None,None,17G,6890
159,RUE JULES HOEBEKE,NaN,worksite,14,6210,137a2b31-437a-4b57-89b0-0224574bef3f,FRASNES LEZ GOSSELIES,NaN,UJn41IoBUOVbPdvwmiBN,NaN,...,"[4.4287067, 50.5351061]",address_interpol,Rue Jules Hoebeke,None,None,Les Bons Villers,None,None,14,6210
282,Rue des Alouettes,720,legal,20,4041,1663c620-635a-47b2-9d56-7f2747037ce2,Herstal,150.0,w24Y1IoBmx0tQqS24FoD,R,...,"[5.5814638, 50.6909204]",address_interpol,Rue des Alouettes,None,None,Herstal,None,None,20/R,4041
541,Rue d'Abhooz,NaN,establishmentUnit,23,4040,d86f0d1c-9f2c-470f-8eb0-7ddac518ba17,Herstal,NaN,jOQd0ooBc6EOG49b_U0P,NaN,...,"[5.620297, 50.6940396]",address_interpol,Rue d'Abhooz,None,None,Herstal,None,None,23,4040
1479,"Rue Ferdinand Nicolay 1, 4102, Seraing, Ougrée",NaN,worksite,1,4102,fd5dbf32-f77b-4f17-9af2-8589b825eaff,Ougrée,NaN,-0L004oBL0HW2X5F1KQc,NaN,...,"[5.5392509, 50.6046678]",address_interpol,Rue Ferdinand Nicolay,None,None,Seraing,None,None,117,4102
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98662,Route de la Malle-Poste,NaN,establishmentUnit,3,4171,d2225d94-5e35-4270-9c3a-fc1477e7f8d2,Comblain-au-Pont,NaN,jEIa04oBL0HW2X5F5HWC,NaN,...,"[5.5504215, 50.500213]",address_interpol,Route de la Malle-Poste,None,None,Comblain-au-Pont,None,None,3,4171
98884,Chaussée de Ghislenghien,NaN,worksite,22,7830,3d9c92de-0915-4d1f-a545-35286a4f33d6,Silly,NaN,Yv_ohY0BTm0LIdLb2CSc,NaN,...,"[3.9204067, 50.6409994]",address_interpol,Chaussée de Ghislenghien,None,None,Silly,None,None,22,7830
98931,"Chaussée de Louvain,H.-M.",NaN,establishmentUnit,44,1320,5e4b8fd1-e0b4-4f2d-be45-1bdd07a1734a,Beauvechain,NaN,ri2cho0BbTjvScB2gC7o,NaN,...,"[4.7146184, 50.7829794]",address_interpol,Chaussée de Louvain,None,None,Beauvechain,None,None,44A,1320
98950,Thier Hamal(EVE),720,legal,25A,4630,c2e2557d-b735-475a-bc1f-fc3166b7e96d,Soumagne,150.0,kW2G0ooBmx0tQqS24_wn,NaN,...,"[5.7038219, 50.6484008]",address_interpol,Thier Hamal,None,None,Soumagne,None,None,25A,4630


In [143]:
# output[output.precision== "city"]
# output

In [144]:
output.to_csv(f"{data_dir}/{output_filename}.gz", index=False)

In [145]:
def print_precision_stats(df):
    vc = df["precision"].value_counts()

    with pd.option_context("display.float_format", '{:,.2%}'.format):
        print(vc/df.shape[0])
    
    print("")
    print(f'building:  {vc[["address", "street_interpol", "address_interpol"]].sum()/df.shape[0]:.2%}')
    print(f'street:    {vc[["street", "address_streetcenter"]].sum()/df.shape[0]:.2%}')
    print(f'city:      {vc[["city"]].sum()/df.shape[0]:.2%}')
    print(f'country:   {vc[[f for f in ["street_00", "address_00", "country"] if f in vc ]].sum()/df.shape[0]:.2%}')
    print(f'NONE:      {vc[["[none]"]].sum()/df.shape[0]:.2%}')

In [146]:
print("Precision stats for all addresses")
print_precision_stats(output)

Precision stats for all addresses
precision
address                77.15%
street                 10.86%
city                    6.89%
street_interpol         3.60%
address_streetcenter    0.55%
street_00               0.46%
address_interpol        0.36%
address_00              0.10%
[none]                  0.02%
Name: count, dtype: float64

building:  81.12%
street:    11.41%
city:      6.89%
country:   0.56%
NONE:      0.02%


In [149]:
print("Precision stats per unique addresses")
print_precision_stats(output.drop_duplicates(subset=field_mapping.keys()))

Precision stats per unique addresses
precision
address                81.80%
street                  8.41%
city                    4.49%
street_interpol         3.78%
address_streetcenter    0.58%
address_interpol        0.45%
street_00               0.38%
address_00              0.10%
[none]                  0.02%
Name: count, dtype: float64

building:  86.03%
street:    8.98%
city:      4.49%
country:   0.48%
NONE:      0.02%


In [150]:
addresses_geocoded[addresses_geocoded.json.isnull()]

,streetName,houseNumber,postCode,municipalityName,json
5127,NaN,1,4852,NaN,None
10347,unknown,unknown,1035,Ministerie van het BHG,None
11752,Tramlijn 10 - 10848W,unknown,1120,Neder-over-Heembeek (Bru.),None
16501,Via Nova,unknown,4980,Fosse (Lg.),None
16725,unknown,unknown,6941,Bomal-sur-Ourthe,None
24500,NaN,65,9230,NaN,None
31819,NaN,141,1330,NaN,None
32880,unknown,unknown,6940,Barvaux-sur-Ourthe,None
40836,NaN,18,3600,NaN,None
42168,unknown,unknown,7742,Hérinnes-lez-Pecq,None
